
## Modifications (per research pipeline review)
- **Keeps ALL channels** (no channel dropping — preserves the full 130-ch HD-EEG montage)
- **Keeps native 512 Hz sampling rate** (no resampling to 250 Hz)
- **Skips resting-state files** (only processes SSAEP and SSVEP conditions)

In [1]:
import os
import re
import glob
import time
import warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except Exception:
    cp = None
    CUPY_AVAILABLE = False

try:
    from mne_icalabel import label_components
    ICALABEL_AVAILABLE = True
except Exception:
    label_components = None
    ICALABEL_AVAILABLE = False

print(f'MNE version: {mne.__version__}')
print(f'CuPy available: {CUPY_AVAILABLE}')
print(f'ICLabel available: {ICALABEL_AVAILABLE}')

if CUPY_AVAILABLE:
    try:
        mne.utils.set_config('MNE_USE_CUDA', 'true', set_env=True)
        mne.cuda.init_cuda(verbose=False)
        USE_GPU = True
        print('CUDA mode enabled.')
    except Exception as exc:
        mne.utils.set_config('MNE_USE_CUDA', 'false', set_env=True)
        USE_GPU = False
        print(f'CUDA initialization failed: {exc}')
else:
    USE_GPU = False
    mne.utils.set_config('MNE_USE_CUDA', 'false', set_env=True)

BASE_DIR = os.getcwd()
DATASET_DIR = os.path.join(BASE_DIR, 'Dataset')
OUTPUT_DIR = os.path.join(BASE_DIR, 'data', 'MIGRAINE_GPU_preprocessed')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Keep native 512 Hz sampling rate (no resampling)
TARGET_SFREQ = 512
LINE_NOISE_FREQ = 50
HIGHPASS_FREQ = 1.0
LOWPASS_FREQ = 100.0
IC_REJECTION_THRESHOLD = 0.80
EPOCH_DURATION = 2.0
EPOCH_OVERLAP = 0.0
AMPLITUDE_REJECT_UV = 250e-6
RANDOM_STATE = 42

BANDS = {
    'delta': (0.5, 3),
    'theta': (4, 7),
    'alpha': (8, 12),
    'beta': (12, 30),
    'gamma': (30, 100),
}

AUX_CHANNELS = ['GSR1', 'GSR2', 'Erg1', 'Erg2', 'Resp', 'Plet', 'Temp']
STIM_CHANNEL = 'Status'
EOG_CHANNELS = ['LO1', 'LO2', 'IO1', 'IO2', 'SO1']
ECG_CHANNEL = 'ECG'
MASTOID_CHNLS = ['M1', 'M2']

print(f'Using GPU: {USE_GPU}')
print(f'Dataset dir: {DATASET_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Target sfreq: {TARGET_SFREQ} Hz (native, no resampling)')

MNE version: 1.11.0
CuPy available: True
ICLabel available: True
CUDA mode enabled.
Using GPU: True
Dataset dir: g:\Study\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\Dataset
Output dir: g:\Study\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\data\MIGRAINE_GPU_preprocessed
Target sfreq: 512 Hz (native, no resampling)


## 2. Dataset Discovery in Dataset/

This section scans the Dataset folder, identifies subject folders, and locates the available BDF files and migraine label text files.

In [2]:
def discover_subjects(dataset_dir):
    subjects = {}
    for subject_dir in sorted(os.listdir(dataset_dir)):
        subject_path = os.path.join(dataset_dir, subject_dir)
        if not os.path.isdir(subject_path):
            continue
        if not re.match(r'^[CM]\d+', subject_dir):
            continue
        bdfs = [
            f for f in glob.glob(os.path.join(subject_path, '**', '*.bdf'), recursive=True)
            if '__MACOSX' not in f and os.path.isfile(f)
        ]
        subjects[subject_dir] = {
            'path': subject_path,
            'bdfs': sorted(bdfs),
            'label_file': None,
        }
        for candidate in glob.glob(os.path.join(subject_path, '*migraine*.txt')):
            subjects[subject_dir]['label_file'] = candidate
            break
    return subjects

subjects = discover_subjects(DATASET_DIR)
print(f'Found {len(subjects)} subject folders')

for subject_id, info in list(subjects.items())[:10]:
    print(subject_id, '->', len(info['bdfs']), 'BDF files', 'label file' if info['label_file'] else 'no label file')


Found 39 subject folders
C1 -> 3 BDF files no label file
C10 -> 3 BDF files no label file
C11 -> 3 BDF files no label file
C12 -> 3 BDF files no label file
C13 -> 3 BDF files no label file
C14 -> 4 BDF files no label file
C15 -> 3 BDF files no label file
C16 -> 3 BDF files no label file
C17 -> 3 BDF files no label file
C18 -> 3 BDF files no label file


## 3. Migraine Label Parsing & Feasibility Check

This section reads the available migraine label files, builds a metadata table, and clarifies what classification and batch-preprocessing tasks are feasible from the available labels.

In [3]:
def parse_label_file(label_file):
    if not label_file or not os.path.exists(label_file):
        return None
    try:
        with open(label_file, 'r', encoding='utf-8', errors='ignore') as fh:
            text = fh.read().strip()
        if not text:
            return None
        values = re.findall(r'-?\d+', text)
        if values:
            return int(values[-1])
        return None
    except Exception:
        return None

label_rows = []
for subject_id, info in subjects.items():
    label_value = parse_label_file(info['label_file'])
    if label_value is None:
        label_value = np.nan
    label_rows.append({
        'subject': subject_id,
        'group': 'migraine' if subject_id.startswith('M') else 'control',
        'label': int(label_value) if not pd.isna(label_value) else np.nan,
        'label_file': info['label_file'],
        'n_bdf_files': len(info['bdfs']),
    })

metadata = pd.DataFrame(label_rows)
print(metadata.head())
print('\nLabel availability summary:')
print(metadata['label'].isna().sum(), 'missing labels out of', len(metadata))


  subject    group  label label_file  n_bdf_files
0      C1  control    NaN       None            3
1     C10  control    NaN       None            3
2     C11  control    NaN       None            3
3     C12  control    NaN       None            3
4     C13  control    NaN       None            3

Label availability summary:
39 missing labels out of 39


In [4]:
print('What is possible:')
print('- Subject-level migraine vs control labels can be derived from the folder naming and available text files.')
print('- Batch preprocessing of all available recordings is feasible.')
print('- Classical ML and lightweight deep-learning experiments are possible with the cleaned epochs.')
print('')
print('What is not reliable or not guaranteed:')
print('- Trial-level or event-specific labels are not guaranteed unless the files explicitly contain them.')
print('- Clinical-grade generalization is not realistic without external validation.')
print('- Small-sample training may overfit, so cross-validation and careful interpretation are required.')

metadata.to_csv(os.path.join(OUTPUT_DIR, 'subject_metadata.csv'), index=False)
print(f'\nSaved metadata to {os.path.join(OUTPUT_DIR, "subject_metadata.csv")}')


What is possible:
- Subject-level migraine vs control labels can be derived from the folder naming and available text files.
- Batch preprocessing of all available recordings is feasible.
- Classical ML and lightweight deep-learning experiments are possible with the cleaned epochs.

What is not reliable or not guaranteed:
- Trial-level or event-specific labels are not guaranteed unless the files explicitly contain them.
- Clinical-grade generalization is not realistic without external validation.
- Small-sample training may overfit, so cross-validation and careful interpretation are required.

Saved metadata to g:\Study\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\data\MIGRAINE_GPU_preprocessed\subject_metadata.csv


## 4. Shared Preprocessing Parameters

These settings follow the same LEMON-style choices for filtering, ICA, and epoching so the migraine data can be processed consistently.

In [5]:
# Keep native 512 Hz sampling rate (no resampling)
TARGET_SFREQ = 512
LINE_NOISE_FREQ = 50
HIGHPASS_FREQ = 1.0
LOWPASS_FREQ = 100.0
IC_REJECTION_THRESHOLD = 0.80
EPOCH_DURATION = 2.0
EPOCH_OVERLAP = 0.0
AMPLITUDE_REJECT_UV = 250e-6
RANDOM_STATE = 42

BANDS = {
    'delta': (0.5, 3),
    'theta': (4, 7),
    'alpha': (8, 12),
    'beta': (12, 30),
    'gamma': (30, 100),
}

print('Preprocessing settings configured for the migraine dataset.')
print(f'Target sfreq: {TARGET_SFREQ} Hz (native, no resampling)')

Preprocessing settings configured for the migraine dataset.
Target sfreq: 512 Hz (native, no resampling)


## 5. Helper Functions for EEG Cleaning



In [6]:
def setup_channels(raw):
    """Keep ALL channels. Re-type non-EEG channels so they are excluded from
    epoching via pick('eeg'), but never drop any channel from the raw."""
   
    
    type_map = {}

    for ch in AUX_CHANNELS:
        if ch in raw.ch_names:
            type_map[ch] = 'misc'
   
    if STIM_CHANNEL in raw.ch_names:
        type_map[STIM_CHANNEL] = 'stim'
    
    for ch in EOG_CHANNELS:
        if ch in raw.ch_names:
            type_map[ch] = 'eog'
    
    if ECG_CHANNEL in raw.ch_names:
        type_map[ECG_CHANNEL] = 'ecg'
   
    for ch in MASTOID_CHNLS:
        if ch in raw.ch_names:
            type_map[ch] = 'eeg'
    if type_map:
        raw.set_channel_types(type_map)

    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage, on_missing='ignore', verbose=False)
    return raw


def detect_bad_channels(raw, random_state=42):
    picks_eeg = mne.pick_types(raw.info, eeg=True, eog=False, exclude=[])
    if len(picks_eeg) < 2:
        return [], False

    data = raw.get_data(picks=picks_eeg) * 1e6
    stds = data.std(axis=1)
    median_std = np.median(stds)
    bad = []
    if median_std > 0:
        bad = [raw.ch_names[p] for p in picks_eeg if (stds[list(picks_eeg).index(p)] < 0.5) or (stds[list(picks_eeg).index(p)] > 5 * median_std)]
    return sorted(set(bad)), False


def annotate_bad_segments(raw, threshold_uv=250, window_s=0.5):
    sfreq = raw.info['sfreq']
    window_samples = int(window_s * sfreq)
    picks_eeg = mne.pick_types(raw.info, eeg=True, eog=False)
    data = raw.get_data(picks=picks_eeg) * 1e6
    n_windows = max(1, data.shape[1] // window_samples)
    bad_annotations = []

    for w in range(n_windows):
        start = w * window_samples
        end = start + window_samples
        window_data = data[:, start:end]
        ptp = np.ptp(window_data, axis=1)
        if np.any(ptp > threshold_uv):
            bad_annotations.append((start / sfreq, window_s, 'BAD_artifact'))

    if bad_annotations:
        onsets = [a[0] for a in bad_annotations]
        durations = [a[1] for a in bad_annotations]
        desc = [a[2] for a in bad_annotations]
        annot = mne.Annotations(onset=onsets, duration=durations, description=desc)
        raw.set_annotations(raw.annotations + annot)

    return len(bad_annotations) * window_s


def filter_band(epochs, l_freq, h_freq):
    return epochs.copy().filter(
        l_freq=l_freq,
        h_freq=h_freq,
        method='fir',
        phase='zero',
        picks='eeg',
        verbose=False,
    )


def run_ica(raw, random_state=42):
    n_eeg = len(mne.pick_types(raw.info, eeg=True, eog=False))
    if n_eeg < 2:
        return raw, []
    n_components = max(1, min(20, n_eeg - 1))
    ica = mne.preprocessing.ICA(
        n_components=n_components,
        method='infomax',
        fit_params=dict(extended=True),
        random_state=random_state,
        max_iter='auto',
        verbose=False,
    )
    ica.fit(raw, picks='eeg', verbose=False)
    if label_components is not None:
        try:
            ic_labels = label_components(raw, ica, method='iclabel')
            labels = ic_labels.get('labels', [])
            probs = np.asarray(ic_labels.get('y_pred_proba', []))
            reject_label_set = {'eye blink', 'muscle artifact', 'heart beat', 'line noise', 'channel noise', 'other'}
            reject_ics = [i for i, label in enumerate(labels) if label in reject_label_set and float(probs[i]) >= IC_REJECTION_THRESHOLD]
            ica.exclude = reject_ics
        except Exception:
            ica.exclude = []
    else:
        ica.exclude = []
    raw_clean = ica.apply(raw, verbose=False)
    return raw_clean, ica.exclude

print('Helper functions defined.')

Helper functions defined.


## 6. Subject-Level Preprocessing Pipeline

This section loads each BDF recording, applies channel setup and filtering, removes artifacts via ICA and bad-channel handling, epochs the data, and saves the cleaned arrays.

**Note:** The pipeline now **skips resting-state files** — only SSAEP and SSVEP conditions are processed.

In [7]:
def infer_condition_name(bdf_file):
    name = os.path.basename(bdf_file).lower()
    if 'rest' in name or 'resting' in name:
        return 'resting'
    if 'ssvep' in name:
        return 'ssvep'
    if 'ssaep' in name:
        return 'ssaep'
    if 'task' in name:
        return 'task'
    return 'other'


def preprocess_subject(subject_id, bdf_file, output_dir, overwrite=False, condition_name=None):
    if condition_name is None:
        condition_name = infer_condition_name(bdf_file)

    cond_dir = os.path.join(output_dir, condition_name)
    os.makedirs(cond_dir, exist_ok=True)

    source_stem = os.path.splitext(os.path.basename(bdf_file))[0]
    safe_source = re.sub(r'[^A-Za-z0-9._-]+', '_', source_stem).strip('_')
    bb_path = os.path.join(cond_dir, f'{subject_id}_{safe_source}_broadband.npy')
    if os.path.exists(bb_path) and not overwrite:
        return {
            'subject': subject_id,
            'status': 'skipped_existing',
            'condition': condition_name,
            'file': os.path.basename(bdf_file),
        }

    result = {
        'subject': subject_id,
        'status': 'success',
        'condition': condition_name,
        'file': os.path.basename(bdf_file),
    }
    try:
        raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=False)
        raw = setup_channels(raw)

        if {'M1', 'M2'}.issubset(raw.ch_names):
            raw.set_eeg_reference(ref_channels=['M1', 'M2'], projection=False, verbose=False)
        else:
            raw.set_eeg_reference('average', projection=False, verbose=False)

        raw.notch_filter(freqs=[LINE_NOISE_FREQ, LINE_NOISE_FREQ * 2], method='spectrum_fit', verbose=False)
        raw.filter(l_freq=HIGHPASS_FREQ, h_freq=LOWPASS_FREQ, method='fir', phase='zero', picks='eeg', verbose=False)

        if USE_GPU:
            raw.load_data()

        bad_chs, _ = detect_bad_channels(raw, random_state=RANDOM_STATE)
        raw.info['bads'] = bad_chs
        if bad_chs:
            raw.interpolate_bads(reset_bads=True, verbose=False)

        raw_clean, reject_ics = run_ica(raw, random_state=RANDOM_STATE)
        result['n_bad_channels'] = len(bad_chs)
        result['n_rejected_ics'] = len(reject_ics)

        bad_s = annotate_bad_segments(raw_clean, threshold_uv=250, window_s=0.5)
        result['bad_segments_s'] = bad_s

        raw_eeg = raw_clean.copy().pick('eeg')
        epochs = mne.make_fixed_length_epochs(raw_eeg, duration=EPOCH_DURATION, overlap=EPOCH_OVERLAP, preload=True, verbose=False)
        n_before = len(epochs)
        epochs.drop_bad(reject=dict(eeg=AMPLITUDE_REJECT_UV), verbose=False)
        result['n_epochs_before'] = n_before
        result['n_epochs_after'] = len(epochs)
        result['epoch_keep_ratio'] = round(len(epochs) / max(1, n_before), 3)

        X = epochs.get_data().astype(np.float32)
        np.save(bb_path, X)
        result['epoch_shape'] = X.shape

        for band, (lf, hf) in BANDS.items():
            ep_band = filter_band(epochs, lf, hf)
            np.save(os.path.join(cond_dir, f'{subject_id}_{safe_source}_{band}.npy'), ep_band.get_data().astype(np.float32))

        with open(os.path.join(cond_dir, f'channel_names_{safe_source}.txt'), 'w') as fh:
            fh.write('\n'.join(epochs.ch_names))

    except Exception as exc:
        result['status'] = 'failed'
        result['error'] = f'{type(exc).__name__}: {exc}'

    return result

print('Subject-level preprocessing function defined.')

Subject-level preprocessing function defined.


## 7. Batch Processing Over All Subjects

This loop walks through the subjects found in the Dataset folder, processes each available recording, and records the preprocessing outcome for every file.

**Note:** Resting-state files are **skipped** — only SSAEP and SSVEP conditions are processed.

In [8]:
results = []
processed_files = 0
skipped_resting = 0

for subject_id, info in sorted(subjects.items()):
    for bdf_file in info['bdfs']:
        condition_name = infer_condition_name(bdf_file)
        
        # Skip resting-state files (only process SSAEP/SSVEP)
        if condition_name == 'resting':
            skipped_resting += 1
            continue
        
        print(f'Processing {subject_id}: {os.path.basename(bdf_file)} [{condition_name}]')
        res = preprocess_subject(subject_id, bdf_file, OUTPUT_DIR, overwrite=False, condition_name=condition_name)
        results.append(res)
        processed_files += 1

        if res.get('status') == 'success':
            print(
                f"  -> bad_channels={res.get('n_bad_channels', 0)}, "
                f"rejected_ics={res.get('n_rejected_ics', 0)}, "
                f"epochs_before={res.get('n_epochs_before', 0)}, "
                f"epochs_after={res.get('n_epochs_after', 0)}"
            )
        else:
            print(f"  -> failed: {res.get('error', 'unknown')}")

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_DIR, 'preprocessing_summary.csv'), index=False)
print(results_df[['subject', 'condition', 'file', 'status', 'n_bad_channels', 'n_rejected_ics', 'n_epochs_before', 'n_epochs_after']].head())
print(f'\nProcessed {processed_files} BDF files (SSAEP/SSVEP only).')
print(f'Skipped {skipped_resting} resting-state files.')

Processing C1: C1_SSAEP.bdf [ssaep]
  -> failed: unknown
Processing C1: C1_SSVEP.bdf [ssvep]
  -> failed: unknown
Processing C10: C10_SSAEP.bdf [ssaep]
  -> failed: unknown
Processing C10: C10_SSVEP.bdf [ssvep]
  -> failed: unknown
Processing C11: C11_SSAEP.bdf [ssaep]
  -> failed: unknown
Processing C11: C11_SSVEP.bdf [ssvep]
  -> failed: unknown
Processing C12: C12_SSAEP.bdf [ssaep]
  -> failed: unknown
Processing C12: C12_SSVEP.bdf [ssvep]
  -> failed: unknown
Processing C13: C13_SSAEP.bdf [ssaep]
  -> failed: unknown
Processing C13: C13_SSVEP.bdf [ssvep]
  -> failed: unknown
Processing C14: C14_SSAEP.bdf [ssaep]
  -> failed: unknown
Processing C14: C14_SSAEP_2.bdf [ssaep]
  -> failed: unknown
Processing C14: C14_SSVEP.bdf [ssvep]
  -> failed: unknown
Processing C15: C15_SSAEP.bdf [ssaep]
  -> failed: unknown
Processing C15: C15_SSVEP.bdf [ssvep]
  -> failed: unknown
Processing C16: C16_SSAEP.bdf [ssaep]
  -> failed: unknown
Processing C16: C16_SSVEP.bdf [ssvep]
  -> failed: unknown

## 8. QC Summary, Saved Outputs, and Model-Ready Dataset Export

This section summarizes preprocessing quality, saves the labels and metadata, and builds stacked arrays for downstream modeling.

**Note:** The stacking section now builds datasets for **both SSAEP and SSVEP** conditions (resting is skipped).

In [ ]:
if not results_df.empty:
    print('QC summary:')
    print(results_df[['status', 'n_bad_channels', 'n_rejected_ics', 'n_epochs_after']].head())

label_rows = []
for subject_id, info in sorted(subjects.items()):
    label_value = parse_label_file(info['label_file'])
    label_rows.append({
        'subject': subject_id,
        'group': 'migraine' if subject_id.startswith('M') else 'control',
        'label': label_value if label_value is not None else (1 if subject_id.startswith('M') else 0),
        'label_file': info['label_file'],
    })

labels_df = pd.DataFrame(label_rows)
labels_df.to_csv(os.path.join(OUTPUT_DIR, 'labels.csv'), index=False)
print(labels_df.head())

# Build stacked datasets for SSAEP and SSVEP conditions (resting is skipped)
for condition_name in ['ssaep', 'ssvep']:
    condition_dir = os.path.join(OUTPUT_DIR, condition_name)
    arrays = []
    y = []

    if os.path.exists(condition_dir):
        for _, row in labels_df.iterrows():
            subject_id = row['subject']
            matching_files = []
            for filename in sorted(os.listdir(condition_dir)):
                if not filename.endswith('_broadband.npy'):
                    continue
                # Exact subject match (avoid C1 matching C10, etc.)
                if re.match(rf'^{re.escape(subject_id)}_', filename):
                    matching_files.append(os.path.join(condition_dir, filename))

            for arr_path in matching_files:
                arr = np.load(arr_path)
                arrays.append(arr)
                y.append(np.full(arr.shape[0], int(row['label']), dtype=np.int64))

    if arrays:
        X = np.concatenate(arrays, axis=0).astype(np.float32)
        y = np.concatenate(y)
        np.savez_compressed(os.path.join(OUTPUT_DIR, f'dataset_{condition_name}.npz'), X=X, y=y)
        print(f'Saved stacked dataset for {condition_name}: shape {X.shape}, labels {y.shape}')
    else:
        print(f'No cleaned arrays were available to stack for {condition_name}.')

QC summary:
             status  n_bad_channels  n_rejected_ics  n_epochs_after
0  skipped_existing             NaN             NaN             NaN
1  skipped_existing             NaN             NaN             NaN
2  skipped_existing             NaN             NaN             NaN
3  skipped_existing             NaN             NaN             NaN
4  skipped_existing             NaN             NaN             NaN
  subject    group  label label_file
0      C1  control      0       None
1     C10  control      0       None
2     C11  control      0       None
3     C12  control      0       None
4     C13  control      0       None
